# Long-Only ML Strategy Notebook

This notebook is organized for a repeatable workflow:
1. Setup
2. Build data/features
3. Train models (pre-2021 cutoff)
4. Build 5-year scoring panel (2021-2025)
5. Define strategy/helpers
6. Run 5-year frozen OOS ablation (model selection)
7. First backtest: Frozen 5-year OOS (no retraining)
8. Run anchored vs frozen 5-year OOS comparison


## A) Setup


In [1]:
import os
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modules.data.context import DataContext
from modules.data.universe_fmp import build_large_liquid_universe_single_call
from modules.api import (
    build_technical_dataframe,
    build_fundamental_dataframe,
    build_macro_dataframe,
    build_label_dataframe,
)
from modules.data.preparation import MLDatasetConfig, prepare_ml_dataset
from modules.workflows import (
    train_rf_models,
    train_ae,
    StrategyCase,
    enrich_scored_panel,
    make_backtest_panel,
    make_exec_cfg,
    run_case as run_eval_case,
    strategy_diagnostics,
    build_anchored_fold,
)
from modules.engine.latest import run_panel_prediction_custom, make_autoencoder_familiarity_predictor
from modules.engine.backtest import backtest_panel, ExecutionConfig

pd.set_option("display.max_columns", 200)

# ------------------------------
# Config
# ------------------------------
TRAIN_CUTOFF = pd.Timestamp("2020-12-31")
BT_START = pd.Timestamp("2021-01-01")
BT_END = pd.Timestamp("2025-12-31")

TOP_K = 5
REBALANCE_FREQ = "W"      # D, W, M
GATE_Q = 0.65              # component gate percentile
GROSS = 0.8                # long-only gross exposure (short budget is 0.0)

FEE_BPS = 5.0
SLIPPAGE_BPS = 5.0

# Runtime controls
CACHE_ENABLED = True
FAST_ABLATION_METRICS = True  # skips expensive per-case weight recomputation in yearly ablation

## B) Build Features


In [2]:
# 1) Context + universe + feature panel
load_dotenv()
FMP_API_KEY = os.getenv("FMP_API_KEY")
if not FMP_API_KEY:
    raise RuntimeError("Missing FMP_API_KEY in environment/.env")

ctx = DataContext.from_data_dir(
    api_key=FMP_API_KEY,
    data_dir="./data",
    db_name="quant.db",
    sleep_s=0.0,
    history_years=100,
)

universe = tuple(build_large_liquid_universe_single_call(
    api_key=FMP_API_KEY,
    marketCapMoreThan=10_000_000_000,
    priceMoreThan=5,
    country="US",
    exchange="NASDAQ,NYSE,AMEX",
    isEtf=False,
    isFund=False,
    includeAllShareClasses=False,
    limit=10000,
))
if len(universe) == 0:
    raise RuntimeError("Screener returned 0 symbols.")

START_DATE = "1980-01-01"
END_DATE = BT_END.strftime("%Y-%m-%d")

technical_df, technical_cols = build_technical_dataframe(
    ctx=ctx,
    symbols=universe,
    start_date=START_DATE,
    end_date=END_DATE,
    verbose_data=False,
)
fund_df, fund_cols = build_fundamental_dataframe(
    ctx=ctx,
    symbols=universe,
    start_date=START_DATE,
    end_date=END_DATE,
    target_index=technical_df.index,
    daily_prices=technical_df,
    verbose=False,
)
macro_df, macro_cols = build_macro_dataframe(
    ctx=ctx,
    start_date=START_DATE,
    end_date=END_DATE,
    target_index=technical_df.index,
    verbose=False,
)

final_df = pd.concat([technical_df, fund_df, macro_df], axis=1).sort_index()

print("Universe size:", len(universe))
print("Feature panel shape:", final_df.shape)
print("Feature date range:", final_df.index.get_level_values("date").min().date(), "->", final_df.index.get_level_values("date").max().date())


[WARNING] Skipped 9 symbols during build.


/Users/johnnylee/PycharmProjects/optimal_trader/modules/features/fundamentals_fmp.py:105: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  fund_df = pd.concat(dfs_per_symbol, ignore_index=True)


Universe size: 785
Feature panel shape: (4005442, 202)
Feature date range: 1997-05-12 -> 2025-12-31


## C) Train Base Models (Pre-2021 Cutoff)


In [3]:
# 2) Train models using data <= TRAIN_CUTOFF (pre-2021)
train_mask = final_df.index.get_level_values("date") <= TRAIN_CUTOFF
features_train = final_df.loc[train_mask].copy()

symbols_in_train = set(technical_df.loc[technical_df.index.get_level_values("date") <= TRAIN_CUTOFF].index.get_level_values("symbol"))
daily_map_train = {
    s: technical_df.xs(s, level="symbol").loc[:TRAIN_CUTOFF].copy()
    for s in universe
    if s in symbols_in_train
}

K_PARAMS = {"YE": [1, 2, 4, 8]}
EXECUTION_PARAMS = {"price_col": "close", "fee_bps": 5.0, "slippage_bps": 5.0}
WEIGHTING_PARAMS = {
    "use_sample_weight": True,
    "alpha": 4.0,
    "r_clip": 0.10,
    "horizon_balance": True,
}

label_df_train = build_label_dataframe(
    daily_by_symbol=daily_map_train,
    k_params=K_PARAMS,
    execution_params=EXECUTION_PARAMS,
    weighting=WEIGHTING_PARAMS,
    add_rank_labels=True,
    verbose=False,
)

train_df, raw_feature_list, _ = prepare_ml_dataset(
    features_df=features_train,
    labels_df=label_df_train,
    target_cols=["label", "trade_return", "market_position", "trade_duration_days"],
    weight_col="sample_weight",
    config=MLDatasetConfig(drop_nan_features=False),
    verbose=True,
)

rf_bundle = train_rf_models(
    train_df,
    raw_feature_list,
    split_ratio=1.0,
    classifier_target_col="label",
    ranking_target_col="rank_y",
    classifier_market_position_col="market_position",
    train_trade_return_model=True,
    trade_return_target_col="trade_return",
    train_duration_model=False,
)

clf_raw = rf_bundle.clf
reg_raw = rf_bundle.trade_return_reg if rf_bundle.trade_return_reg is not None else rf_bundle.ranking_reg

ae_raw, ae_numeric_cols = train_ae(train_df, raw_feature_list)

print("Train rows:", len(train_df))
print("Train date max:", pd.to_datetime(train_df.index.get_level_values('date')).max().date())


--- Preparing ML Training Dataset ---
  - Final Training Rows: 396,296
  - Active Features:     202
  - Targets:             ['label', 'trade_return', 'market_position', 'trade_duration_days']
  - Sample Weight Col:   sample_weight

  DIAGNOSTIC: SKLEARN RANDOM FOREST CLASSIFIER (TEST RESULTS)
DATASET & SPLIT:
  - Total Observations: 396,296
  - Split Mode:         In-sample eval (no internal holdout split)
  - Features:           198 numeric (filtered 5 strings)

CLASS DISTRIBUTION (Mapping: {0: 'buy', 1: 'short', 2: 'sell', 3: 'cover'}):
               Train Set              Test Set
  -      buy:  99,707 (25.2%)    99,707 (25.2%)
  -    short:  98,443 (24.8%)    98,443 (24.8%)
  -     sell:  99,719 (25.2%)    99,719 (25.2%)
  -    cover:  98,427 (24.8%)    98,427 (24.8%)

TEST PERFORMANCE:
  - Accuracy:           98.72%

CONFUSION MATRIX (Test Set):
            Pred     buy Pred   short Pred    sell Pred   cover 
True     buy:        97471         2236            0            0 
Tru

## D) Build 5-Year Scoring Panel (2021-2025)


In [4]:
# 3) Score panel for 2021-2025 window
panel_5y = final_df.loc[
    (final_df.index.get_level_values("date") >= BT_START)
    & (final_df.index.get_level_values("date") <= BT_END)
].copy()

if panel_5y.empty:
    raise RuntimeError("No feature rows in 5-year panel (2021-2025).")

ae_predict = make_autoencoder_familiarity_predictor(ae_numeric_cols)

scored_panel = run_panel_prediction_custom(
    train_data=panel_5y,
    model_specs=[
        {"model": clf_raw, "pred_col": "clf", "include_class_probs": True},
        {"model": reg_raw, "pred_col": "ranking"},
        {"model": ae_raw, "pred_col": "ae_familiarity", "predict_fn": lambda df, m: ae_predict(df, m)},
    ],
    market_position_value=0,
    combine_scores_fn=lambda df: pd.to_numeric(df.get("clf__prob_buy", 0.0), errors="coerce").fillna(0.0)
    * pd.to_numeric(df.get("ranking", 0.0), errors="coerce").fillna(0.0)
    * pd.to_numeric(df.get("ae_familiarity", 1.0), errors="coerce").fillna(1.0),
    row_filter_fn=None,
    round_decimals=None,
)

scored_panel = enrich_scored_panel(scored_panel)
bt_panel_5y = make_backtest_panel(
    scored_panel=scored_panel,
    technical_df=technical_df,
    start=BT_START,
    end=BT_END,
)

print("Backtest panel shape (5-year):", bt_panel_5y.shape)
print("Backtest date range:", bt_panel_5y.index.get_level_values("date").min().date(), "->", bt_panel_5y.index.get_level_values("date").max().date())


Backtest panel shape (5-year): (888758, 217)
Backtest date range: 2021-01-04 -> 2025-12-31


## E) Strategy And Backtest Helpers (Shared)


In [5]:
# 4) Strategy + backtest helpers

cfg = make_exec_cfg(fee_bps=FEE_BPS, slippage_bps=SLIPPAGE_BPS)


def run_case(*, case_name: str, top_k: int, gate_q: float, long_budget: float, short_budget: float,
             use_vol_scaling: bool, min_weight_change: float, rebalance_freq: str = "W", hold_gate_q: float = 0.50,
             hold_drop_pct: float | None = None):
    case = StrategyCase(
        case_name=case_name,
        top_k=int(top_k),
        gate_q=float(gate_q),
        hold_gate_q=float(hold_gate_q),
        hold_drop_pct=hold_drop_pct,
        long_budget=float(long_budget),
        short_budget=float(short_budget),
        use_vol_scaling=bool(use_vol_scaling),
        min_weight_change=float(min_weight_change),
        rebalance_freq=str(rebalance_freq),
    )
    return run_eval_case(clf_model=clf_raw, panel=bt_panel_5y, case=case, exec_cfg=cfg)


## F) Diagnostics Helper


In [6]:
# 5) Diagnostics helper

def show_strategy_diagnostics(strategy_obj, panel_obj):
    diag_df = strategy_diagnostics(strategy_obj, panel_obj)
    display(diag_df.tail(15))
    print("Summary:")
    display(diag_df.describe())

    ax = diag_df[["n_long", "n_short"]].plot(figsize=(12, 4), title="Active names per day")
    ax.set_ylabel("Count")
    ax.grid(True, alpha=0.3)
    plt.show()

    ax = diag_df[["gross_exposure", "turnover_est"]].plot(figsize=(12, 4), title="Gross exposure and estimated turnover")
    ax.grid(True, alpha=0.3)
    plt.show()


## G) 5-Year Frozen OOS Ablation (Run First)


In [7]:
# 6) Ablation study first (select best config)
from itertools import product
import contextlib
import io

# Keep this grid aligned with the year-by-year stability study
INCLUDE_BALANCED_CASES = False
MAX_CASES = None  # set to int (e.g., 120) for faster local screening
RANDOM_SEED = 42

# Big parameter moves only (coarse exploration)
# Fixed low-impact knobs based on prior runs:
# - vol_scaling fixed to True
# - min_weight_change fixed to 0.0
# - rebalance fixed to weekly
TOP_K_GRID = [1, 5, 10, 20]
GATE_Q_GRID = [0.50, 0.80]          # entry gate
HOLD_GATE_Q_GRID = [0.50]           # fixed hold gate
HOLD_DROP_PCT_GRID = [0.10, 0.20, 0.30]  # exit when score drops from entry
LONG_BUDGET_GRID = [0.4, 0.8, 1.0]
REBALANCE_GRID = ["W"]
VOL_SCALING_GRID = [True]
MIN_WEIGHT_CHANGE_GRID = [0.0]

# Evaluate ablation on 5-year OOS with frozen model (train once at first-year cutoff)
ABLATION_YEARS = [2021, 2022, 2023, 2024, 2025]
ABLATION_ANCHOR_TRAIN_START = pd.Timestamp("1980-01-01")


def _build_fold_quiet_for_ablation(test_year: int):
    with contextlib.redirect_stdout(io.StringIO()):
        return build_anchored_fold(
            test_year=int(test_year),
            anchor_train_start=ABLATION_ANCHOR_TRAIN_START,
            universe=universe,
            technical_df=technical_df,
            final_df=final_df,
            k_params=K_PARAMS,
            execution_params=EXECUTION_PARAMS,
            weighting_params=WEIGHTING_PARAMS,
        )


ablation_bt_panels = []
bt_first_ablation, clf_ablation_frozen, train_df_ablation, cutoff_ablation, _, _ = _build_fold_quiet_for_ablation(ABLATION_YEARS[0])
ablation_bt_panels.append(bt_first_ablation)
for yr in ABLATION_YEARS[1:]:
    bt_i, _, _, _, _, _ = _build_fold_quiet_for_ablation(yr)
    ablation_bt_panels.append(bt_i)

ablation_bt_panel_5y = pd.concat(ablation_bt_panels, axis=0).sort_index()
ablation_bt_panel_5y = ablation_bt_panel_5y.loc[~ablation_bt_panel_5y.index.duplicated(keep="first")]


def run_case_five_year(*, case_name: str, top_k: int, gate_q: float, long_budget: float, short_budget: float,
                       use_vol_scaling: bool, min_weight_change: float, rebalance_freq: str = "W", hold_gate_q: float = 0.50,
                       hold_drop_pct: float | None = None):
    case = StrategyCase(
        case_name=case_name,
        top_k=int(top_k),
        gate_q=float(gate_q),
        hold_gate_q=float(hold_gate_q),
        hold_drop_pct=hold_drop_pct,
        long_budget=float(long_budget),
        short_budget=float(short_budget),
        use_vol_scaling=bool(use_vol_scaling),
        min_weight_change=float(min_weight_change),
        rebalance_freq=str(rebalance_freq),
    )
    return run_eval_case(clf_model=clf_ablation_frozen, panel=ablation_bt_panel_5y, case=case, exec_cfg=cfg)

ABLATION_CASES = []
for top_k, gate_q, hold_q, hold_drop, lb, freq, use_vol, mwc in product(
    TOP_K_GRID,
    GATE_Q_GRID,
    HOLD_GATE_Q_GRID,
    HOLD_DROP_PCT_GRID,
    LONG_BUDGET_GRID,
    REBALANCE_GRID,
    VOL_SCALING_GRID,
    MIN_WEIGHT_CHANGE_GRID,
):
    ABLATION_CASES.append(dict(
        case_name=f"long_k{top_k}_eq{gate_q:.2f}_hq{hold_q:.2f}_hd{hold_drop:.2f}_lb{lb:.1f}_{freq}_{'vol' if use_vol else 'eq'}_buf{mwc:.3f}",
        top_k=int(top_k),
        gate_q=float(gate_q),
        hold_gate_q=float(hold_q),
        hold_drop_pct=float(hold_drop),
        long_budget=float(lb),
        short_budget=0.0,
        use_vol_scaling=bool(use_vol),
        min_weight_change=float(mwc),
        rebalance_freq=str(freq),
    ))

if INCLUDE_BALANCED_CASES:
    for top_k, gate_q, hold_q, hold_drop, gross, freq, use_vol, mwc in product(
        TOP_K_GRID,
        GATE_Q_GRID,
        HOLD_GATE_Q_GRID,
        HOLD_DROP_PCT_GRID,
        LONG_BUDGET_GRID,
        REBALANCE_GRID,
        VOL_SCALING_GRID,
        MIN_WEIGHT_CHANGE_GRID,
    ):
        half = float(gross) / 2.0
        ABLATION_CASES.append(dict(
            case_name=f"bal_k{top_k}_eq{gate_q:.2f}_hq{hold_q:.2f}_hd{hold_drop:.2f}_g{gross:.1f}_{freq}_{'vol' if use_vol else 'eq'}_buf{mwc:.3f}",
            top_k=int(top_k),
            gate_q=float(gate_q),
            hold_gate_q=float(hold_q),
            hold_drop_pct=float(hold_drop),
            long_budget=half,
            short_budget=half,
            use_vol_scaling=bool(use_vol),
            min_weight_change=float(mwc),
            rebalance_freq=str(freq),
        ))

if MAX_CASES is not None and len(ABLATION_CASES) > int(MAX_CASES):
    rng = np.random.default_rng(RANDOM_SEED)
    idx = rng.choice(len(ABLATION_CASES), size=int(MAX_CASES), replace=False)
    ABLATION_CASES = [ABLATION_CASES[i] for i in sorted(idx)]

print(
    f"Ablation cases={len(ABLATION_CASES)} | years={ABLATION_YEARS[0]}-{ABLATION_YEARS[-1]} "
    f"| train_cutoff={cutoff_ablation.date()} | train_rows={len(train_df_ablation)} "
    f"| INCLUDE_BALANCED_CASES={INCLUDE_BALANCED_CASES} | MAX_CASES={MAX_CASES}"
)

rows = []
for c in ABLATION_CASES:
    _, _, row = run_case_five_year(**c)
    rows.append(row)

ablation_df = pd.DataFrame(rows).sort_values(["sharpe", "total_return_pct"], ascending=[False, False]).reset_index(drop=True)
print("Ablation results (ranked by Sharpe):")
display(ablation_df[[
    "case", "total_return_pct", "sharpe", "max_drawdown_pct", "avg_turnover",
    "avg_gross_exposure", "median_gross_exposure", "avg_active_names",
    "top_k", "gate_q", "hold_gate_q", "hold_drop_pct", "gross_target", "rebalance", "vol_scaling", "min_weight_change"
]])

BEST_ABLATION = ablation_df.iloc[0].to_dict()
print("Best ablation config:")
display(pd.Series(BEST_ABLATION))


Ablation cases=90 | years=2021-2025 | train_cutoff=2020-12-31 | train_rows=396296 | INCLUDE_BALANCED_CASES=False | MAX_CASES=None
Ablation results (ranked by Sharpe):


,case,total_return_pct,sharpe,max_drawdown_pct,avg_turnover,avg_gross_exposure,median_gross_exposure,avg_active_names,top_k,gate_q,hold_gate_q,hold_drop_pct,gross_target,rebalance,vol_scaling,min_weight_change
0,long_k10_eq0.50_hq0.50_hd0.10_lb1.0_W_vol_buf0...,680.700961,1.491018,-25.364909,0.184194,1.000000,1.0,10.000000,10,0.5,0.5,0.1,1.0,W,True,0.0
1,long_k10_eq0.50_hq0.50_hd0.20_lb1.0_W_vol_buf0...,680.700961,1.491018,-25.364909,0.184194,1.000000,1.0,10.000000,10,0.5,0.5,0.2,1.0,W,True,0.0
2,long_k10_eq0.50_hq0.50_hd0.30_lb1.0_W_vol_buf0...,680.700961,1.491018,-25.364909,0.184194,1.000000,1.0,10.000000,10,0.5,0.5,0.3,1.0,W,True,0.0
3,long_k10_eq0.50_hq0.50_hd0.10_lb0.8_W_vol_buf0...,437.536330,1.491018,-20.379407,0.147355,0.800000,0.8,10.000000,10,0.5,0.5,0.1,0.8,W,True,0.0
4,long_k10_eq0.50_hq0.50_hd0.20_lb0.8_W_vol_buf0...,437.536330,1.491018,-20.379407,0.147355,0.800000,0.8,10.000000,10,0.5,0.5,0.2,0.8,W,True,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,long_k10_eq0.80_hq0.50_hd0.10_lb0.4_W_vol_buf0...,51.168469,0.647946,-17.444987,0.068490,0.353147,0.4,2.667729,10,0.8,0.5,0.1,0.4,W,True,0.0
86,long_k10_eq0.80_hq0.50_hd0.10_lb1.0_W_vol_buf0...,131.464015,0.647946,-39.585365,0.171225,0.882869,1.0,2.667729,10,0.8,0.5,0.1,1.0,W,True,0.0
87,long_k10_eq0.80_hq0.50_hd0.30_lb0.8_W_vol_buf0...,98.507907,0.619997,-34.990788,0.134745,0.712032,0.8,2.720319,10,0.8,0.5,0.3,0.8,W,True,0.0
88,long_k10_eq0.80_hq0.50_hd0.30_lb0.4_W_vol_buf0...,48.442912,0.619997,-18.884653,0.067372,0.356016,0.4,2.720319,10,0.8,0.5,0.3,0.4,W,True,0.0


Best ablation config:


case                     long_k10_eq0.50_hq0.50_hd0.10_lb1.0_W_vol_buf0...
top_k                                                                   10
gate_q                                                                 0.5
hold_gate_q                                                            0.5
long_budget                                                            1.0
short_budget                                                           0.0
gross_target                                                           1.0
rebalance                                                                W
vol_scaling                                                           True
min_weight_change                                                      0.0
hold_drop_pct                                                          0.1
avg_gross_exposure                                                     1.0
median_gross_exposure                                                  1.0
avg_active_names         

## H) First Backtest: Frozen 5-Year OOS Using Best Ablation Config


In [ ]:
# 7) First backtest = Frozen 5-year OOS (train once, no retraining)

FROZEN_BT_YEARS = [2021, 2022, 2023, 2024, 2025]
FROZEN_BT_ANCHOR_TRAIN_START = pd.Timestamp("1980-01-01")


def _build_fold_quiet_for_first_backtest(test_year: int):
    import contextlib
    import io

    with contextlib.redirect_stdout(io.StringIO()):
        return build_anchored_fold(
            test_year=int(test_year),
            anchor_train_start=FROZEN_BT_ANCHOR_TRAIN_START,
            universe=universe,
            technical_df=technical_df,
            final_df=final_df,
            k_params=K_PARAMS,
            execution_params=EXECUTION_PARAMS,
            weighting_params=WEIGHTING_PARAMS,
        )


case_best = StrategyCase(
    case_name=str(BEST_ABLATION.get("case", "best")),
    top_k=int(BEST_ABLATION["top_k"]),
    gate_q=float(BEST_ABLATION["gate_q"]),
    long_budget=float(BEST_ABLATION["long_budget"]),
    short_budget=float(BEST_ABLATION["short_budget"]),
    use_vol_scaling=bool(BEST_ABLATION["vol_scaling"]),
    min_weight_change=float(BEST_ABLATION["min_weight_change"]),
    rebalance_freq=str(BEST_ABLATION["rebalance"]),
)

# Train once at first-year cutoff, then freeze classifier for all OOS years.
bt_first, clf_frozen, train_df_frozen, cutoff_first, _, _ = _build_fold_quiet_for_first_backtest(FROZEN_BT_YEARS[0])

frozen_rows = []
frozen_returns = []
last_strategy = None
last_res = None
last_bt_panel = None

for yr in FROZEN_BT_YEARS:
    bt_i, _, _, _, start_i, end_i = _build_fold_quiet_for_first_backtest(yr)
    strat_i, res_i, _ = run_eval_case(clf_model=clf_frozen, panel=bt_i, case=case_best, exec_cfg=cfg)

    frozen_rows.append({
        "mode": "frozen_no_retrain",
        "test_year": int(yr),
        "anchor_train_start": FROZEN_BT_ANCHOR_TRAIN_START.date().isoformat(),
        "frozen_train_cutoff": cutoff_first.date().isoformat(),
        "frozen_train_rows": int(len(train_df_frozen)),
        "test_start": start_i.date().isoformat(),
        "test_end": end_i.date().isoformat(),
        **res_i.stats,
    })
    frozen_returns.append(res_i.returns.rename(f"ret_frozen_{yr}"))
    last_strategy, last_res, last_bt_panel = strat_i, res_i, bt_i

frozen_df = pd.DataFrame(frozen_rows).sort_values("test_year").reset_index(drop=True)
print("Frozen yearly OOS results (first backtest, no retraining):")
display(frozen_df)

frozen_combined_ret = pd.concat(frozen_returns, axis=0).sort_index()
frozen_combined_eq = (1.0 + frozen_combined_ret).cumprod()
frozen_total_return_pct = float((frozen_combined_eq.iloc[-1] - 1.0) * 100.0) if len(frozen_combined_eq) else np.nan
frozen_sharpe = float((frozen_combined_ret.mean() / frozen_combined_ret.std(ddof=0)) * np.sqrt(252.0)) if frozen_combined_ret.std(ddof=0) > 1e-12 else np.nan
frozen_mdd = float((((frozen_combined_eq / frozen_combined_eq.cummax()) - 1.0).min()) * 100.0) if len(frozen_combined_eq) else np.nan

first_backtest_summary = pd.DataFrame([{
    "mode": "frozen_no_retrain",
    "years": f"{FROZEN_BT_YEARS[0]}-{FROZEN_BT_YEARS[-1]}",
    "combined_total_return_pct": frozen_total_return_pct,
    "combined_sharpe": frozen_sharpe,
    "combined_max_drawdown_pct": frozen_mdd,
}])
print("Frozen combined OOS summary (first backtest):")
display(first_backtest_summary)

fig, ax = plt.subplots(figsize=(12, 4))
frozen_combined_eq.plot(ax=ax, lw=2, color="navy")
ax.set_title(f"Frozen 5-Year OOS Equity ({FROZEN_BT_YEARS[0]}-{FROZEN_BT_YEARS[-1]}) | Best Ablation: {case_best.case_name}")
ax.set_xlabel("Date")
ax.set_ylabel("Equity")
ax.grid(True, alpha=0.3)
plt.show()

display(pd.DataFrame({
    "equity": frozen_combined_eq,
    "returns": frozen_combined_ret,
}).tail())

# Keep compatibility with downstream cells expecting these names.
strategy = last_strategy
res = last_res
best_row = {"case": case_best.case_name}
bt_panel = last_bt_panel

show_strategy_diagnostics(strategy, bt_panel)


## I) Anchored Vs Frozen 5-Year OOS Comparison\n\nAnchored 5-year walk-forward (retrain each year) plus frozen no-retrain benchmark.\n

In [ ]:
# 8) Anchored walk-forward OOS (5 years: 2021-2025)
# Anchored = training always starts at ANCHOR_TRAIN_START and expands each fold.

WF_YEARS = [2021, 2022, 2023, 2024, 2025]
ANCHOR_TRAIN_START = pd.Timestamp("1980-01-01")
WF_CFG = {
    "top_k": 5,
    "gate_q": 0.55,
    "long_budget": 0.8,
    "short_budget": 0.0,
    "use_vol_scaling": True,
    "min_weight_change": 0.01,
    "rebalance_freq": "W",
}
WF_SUPPRESS_MODEL_DIAGNOSTICS = True
WF_CACHE = {}


def _build_anchored_fold_quiet(test_year: int):
    import contextlib
    import io

    if WF_SUPPRESS_MODEL_DIAGNOSTICS:
        with contextlib.redirect_stdout(io.StringIO()):
            return build_anchored_fold(
                test_year=test_year,
                anchor_train_start=ANCHOR_TRAIN_START,
                universe=universe,
                technical_df=technical_df,
                final_df=final_df,
                k_params=K_PARAMS,
                execution_params=EXECUTION_PARAMS,
                weighting_params=WEIGHTING_PARAMS,
            )
    return build_anchored_fold(
        test_year=test_year,
        anchor_train_start=ANCHOR_TRAIN_START,
        universe=universe,
        technical_df=technical_df,
        final_df=final_df,
        k_params=K_PARAMS,
        execution_params=EXECUTION_PARAMS,
        weighting_params=WEIGHTING_PARAMS,
    )


def _get_fold(test_year: int):
    key = (int(test_year), str(ANCHOR_TRAIN_START.date()))
    if CACHE_ENABLED and key in WF_CACHE:
        return WF_CACHE[key]

    fold = _build_anchored_fold_quiet(test_year)
    if CACHE_ENABLED:
        WF_CACHE[key] = fold
    return fold


def _build_bt_panel_for_year_anchored(*, test_year: int, use_cache: bool = True):
    if use_cache:
        return _get_fold(test_year)
    return _build_anchored_fold_quiet(test_year)


def _run_eval_for_panel(*, clf_model, bt_i):
    case_i = StrategyCase(
        case_name="wf",
        top_k=int(WF_CFG["top_k"]),
        gate_q=float(WF_CFG["gate_q"]),
        long_budget=float(WF_CFG["long_budget"]),
        short_budget=float(WF_CFG["short_budget"]),
        use_vol_scaling=bool(WF_CFG["use_vol_scaling"]),
        min_weight_change=float(WF_CFG["min_weight_change"]),
        rebalance_freq=str(WF_CFG["rebalance_freq"]),
    )
    return run_eval_case(clf_model=clf_model, panel=bt_i, case=case_i, exec_cfg=cfg)


# ------------------------------
# A) Anchored (retrain each year)
# ------------------------------
wf_rows = []
wf_returns = []
for yr in WF_YEARS:
    bt_i, clf_i, train_df_i, cutoff_i, start_i, end_i = _build_bt_panel_for_year_anchored(test_year=yr, use_cache=True)
    strat_i, res_i, _ = _run_eval_for_panel(clf_model=clf_i, bt_i=bt_i)

    wf_rows.append({
        "mode": "anchored_retrain",
        "test_year": yr,
        "anchor_train_start": ANCHOR_TRAIN_START.date().isoformat(),
        "train_cutoff": cutoff_i.date().isoformat(),
        "train_rows": int(len(train_df_i)),
        "test_start": start_i.date().isoformat(),
        "test_end": end_i.date().isoformat(),
        **res_i.stats,
    })
    wf_returns.append(res_i.returns.rename(f"ret_{yr}"))

wf_df = pd.DataFrame(wf_rows).sort_values("test_year").reset_index(drop=True)
print("Anchored walk-forward yearly OOS results (retrain each year):")
display(wf_df)

combined_ret = pd.concat(wf_returns, axis=0).sort_index()
combined_eq = (1.0 + combined_ret).cumprod()
combined_total_return_pct = float((combined_eq.iloc[-1] - 1.0) * 100.0) if len(combined_eq) else np.nan
combined_sharpe = float((combined_ret.mean() / combined_ret.std(ddof=0)) * np.sqrt(252.0)) if combined_ret.std(ddof=0) > 1e-12 else np.nan
combined_mdd = float((((combined_eq / combined_eq.cummax()) - 1.0).min()) * 100.0) if len(combined_eq) else np.nan

summary = pd.DataFrame([{
    "mode": "anchored_retrain",
    "years": f"{WF_YEARS[0]}-{WF_YEARS[-1]}",
    "combined_total_return_pct": combined_total_return_pct,
    "combined_sharpe": combined_sharpe,
    "combined_max_drawdown_pct": combined_mdd,
}])
print("Anchored combined OOS summary:")
display(summary)

fig, ax = plt.subplots(figsize=(12, 4))
combined_eq.plot(ax=ax, lw=2, color="darkgreen")
ax.set_title(f"Anchored 5-Year Walk-Forward OOS Equity ({WF_YEARS[0]}-{WF_YEARS[-1]})")
ax.set_ylabel("Equity")
ax.grid(True, alpha=0.3)
plt.show()


# ------------------------------
# B) Frozen test (train once, no retraining)
# ------------------------------
FROZEN_TEST_YEARS = 5
FROZEN_FIRST_YEAR = WF_YEARS[0]
FROZEN_YEARS = [FROZEN_FIRST_YEAR + i for i in range(FROZEN_TEST_YEARS)]
if FROZEN_YEARS[-1] > WF_YEARS[-1]:
    raise ValueError(f"Frozen window {FROZEN_YEARS[0]}-{FROZEN_YEARS[-1]} exceeds available WF_YEARS range {WF_YEARS[0]}-{WF_YEARS[-1]}.")
frozen_train_cutoff = pd.Timestamp(f"{FROZEN_FIRST_YEAR - 1}-12-31")

# Build one fold at first year to get frozen model
bt_first, clf_frozen, train_df_frozen, cutoff_first, _, _ = _build_bt_panel_for_year_anchored(
    test_year=FROZEN_FIRST_YEAR,
    use_cache=True,
)

frozen_rows = []
frozen_returns = []
for yr in FROZEN_YEARS:
    # Reuse feature panel for each year, but keep model frozen at first cutoff
    bt_i, _, _, _, start_i, end_i = _build_bt_panel_for_year_anchored(test_year=yr, use_cache=True)
    strat_i, res_i, _ = _run_eval_for_panel(clf_model=clf_frozen, bt_i=bt_i)

    frozen_rows.append({
        "mode": "frozen_no_retrain",
        "test_year": yr,
        "frozen_train_cutoff": frozen_train_cutoff.date().isoformat(),
        "frozen_train_rows": int(len(train_df_frozen)),
        "test_start": start_i.date().isoformat(),
        "test_end": end_i.date().isoformat(),
        **res_i.stats,
    })
    frozen_returns.append(res_i.returns.rename(f"ret_frozen_{yr}"))

frozen_df = pd.DataFrame(frozen_rows).sort_values("test_year").reset_index(drop=True)
print("Frozen yearly OOS results (no retraining):")
display(frozen_df)

frozen_combined_ret = pd.concat(frozen_returns, axis=0).sort_index()
frozen_combined_eq = (1.0 + frozen_combined_ret).cumprod()
frozen_total_return_pct = float((frozen_combined_eq.iloc[-1] - 1.0) * 100.0) if len(frozen_combined_eq) else np.nan
frozen_sharpe = float((frozen_combined_ret.mean() / frozen_combined_ret.std(ddof=0)) * np.sqrt(252.0)) if frozen_combined_ret.std(ddof=0) > 1e-12 else np.nan
frozen_mdd = float((((frozen_combined_eq / frozen_combined_eq.cummax()) - 1.0).min()) * 100.0) if len(frozen_combined_eq) else np.nan

frozen_summary = pd.DataFrame([{
    "mode": "frozen_no_retrain",
    "years": f"{FROZEN_YEARS[0]}-{FROZEN_YEARS[-1]}",
    "combined_total_return_pct": frozen_total_return_pct,
    "combined_sharpe": frozen_sharpe,
    "combined_max_drawdown_pct": frozen_mdd,
}])
print("Frozen combined OOS summary:")
display(frozen_summary)

# Compare anchored vs frozen
compare_summary = pd.concat([summary, frozen_summary], axis=0, ignore_index=True)
print("Anchored retrain vs Frozen no-retrain:")
display(compare_summary)

fig, ax = plt.subplots(figsize=(12, 4))
combined_eq.plot(ax=ax, lw=2, label="Anchored retrain", color="darkgreen")
frozen_combined_eq.plot(ax=ax, lw=2, label="Frozen no-retrain", color="darkorange")
ax.set_title(f"OOS Equity Comparison ({WF_YEARS[0]}-{WF_YEARS[-1]})")
ax.set_ylabel("Equity")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


## J) Year-By-Year Stability Check (Anchored OOS, 2021-2025)\n\nRun ablation separately for each OOS year to evaluate parameter stability.\n

In [ ]:
# 9) Ablation per OOS year (anchored) - wider parameter search
# Requires Cell #8 to have run so _build_bt_panel_for_year_anchored is available.

if "_build_bt_panel_for_year_anchored" not in globals():
    raise RuntimeError("Run Cell #8 (Anchored walk-forward OOS) first.")

from itertools import product

# ------------------------------
# Search controls (single explicit grid)
# ------------------------------
INCLUDE_BALANCED_CASES = False
MAX_CASES_PER_YEAR = None  # set to int (e.g., 80) if runtime is too long
RANDOM_SEED = 42

# Big parameter moves only (coarse exploration)
# Fixed low-impact knobs based on prior runs:
# - vol_scaling fixed to True
# - min_weight_change fixed to 0.0
# - rebalance fixed to weekly
TOP_K_GRID = [1, 5, 10, 20, 40]
GATE_Q_GRID = [0.50, 0.80]          # entry gate
HOLD_GATE_Q_GRID = [0.50]           # fixed hold gate
HOLD_DROP_PCT_GRID = [0.10, 0.20, 0.30]  # exit when score drops from entry
LONG_BUDGET_GRID = [0.4, 0.8, 1.0]
REBALANCE_GRID = ["W"]
VOL_SCALING_GRID = [True]
MIN_WEIGHT_CHANGE_GRID = [0.0]


def _build_case_grid() -> list[StrategyCase]:
    cases = []
    for top_k, gate_q, hold_q, hold_drop, lb, freq, use_vol, mwc in product(
        TOP_K_GRID,
        GATE_Q_GRID,
        HOLD_GATE_Q_GRID,
        HOLD_DROP_PCT_GRID,
        LONG_BUDGET_GRID,
        REBALANCE_GRID,
        VOL_SCALING_GRID,
        MIN_WEIGHT_CHANGE_GRID,
    ):
        cname = f"long_k{top_k}_eq{gate_q:.2f}_hq{hold_q:.2f}_hd{hold_drop:.2f}_lb{lb:.1f}_{freq}_{'vol' if use_vol else 'eq'}_buf{mwc:.3f}"
        cases.append(
            StrategyCase(
                case_name=cname,
                top_k=int(top_k),
                gate_q=float(gate_q),
                hold_gate_q=float(hold_q),
                hold_drop_pct=float(hold_drop),
                long_budget=float(lb),
                short_budget=0.0,
                use_vol_scaling=bool(use_vol),
                min_weight_change=float(mwc),
                rebalance_freq=str(freq),
            )
        )

    if INCLUDE_BALANCED_CASES:
        for top_k, gate_q, hold_q, hold_drop, gross, freq, use_vol, mwc in product(
            TOP_K_GRID,
            GATE_Q_GRID,
            HOLD_GATE_Q_GRID,
            HOLD_DROP_PCT_GRID,
            LONG_BUDGET_GRID,
            REBALANCE_GRID,
            VOL_SCALING_GRID,
            MIN_WEIGHT_CHANGE_GRID,
        ):
            half = float(gross) / 2.0
            cname = f"bal_k{top_k}_eq{gate_q:.2f}_hq{hold_q:.2f}_hd{hold_drop:.2f}_g{gross:.1f}_{freq}_{'vol' if use_vol else 'eq'}_buf{mwc:.3f}"
            cases.append(
                StrategyCase(
                    case_name=cname,
                    top_k=int(top_k),
                    gate_q=float(gate_q),
                    hold_gate_q=float(hold_q),
                    hold_drop_pct=float(hold_drop),
                    long_budget=half,
                    short_budget=half,
                    use_vol_scaling=bool(use_vol),
                    min_weight_change=float(mwc),
                    rebalance_freq=str(freq),
                )
            )
    return cases


ABLT_CASES_YEARLY = _build_case_grid()
if MAX_CASES_PER_YEAR is not None and len(ABLT_CASES_YEARLY) > int(MAX_CASES_PER_YEAR):
    rng = np.random.default_rng(RANDOM_SEED)
    idx = rng.choice(len(ABLT_CASES_YEARLY), size=int(MAX_CASES_PER_YEAR), replace=False)
    ABLT_CASES_YEARLY = [ABLT_CASES_YEARLY[i] for i in sorted(idx)]

print(
    f"cases/year={len(ABLT_CASES_YEARLY)} "
    f"| INCLUDE_BALANCED_CASES={INCLUDE_BALANCED_CASES} | MAX_CASES_PER_YEAR={MAX_CASES_PER_YEAR}"
)


def _run_case_on_panel(*, clf_model, panel_obj, case: StrategyCase):
    import inspect as _inspect
    from modules.workflows.evaluation import run_case as _wf_run_case

    # Always call the module implementation directly to avoid notebook name collisions.
    fn = _wf_run_case
    kwargs = dict(
        clf_model=clf_model,
        panel=panel_obj,
        case=case,
        exec_cfg=cfg,
    )
    if "compute_weight_metrics" in _inspect.signature(fn).parameters:
        kwargs["compute_weight_metrics"] = (not FAST_ABLATION_METRICS)

    try:
        strat, res, row = fn(**kwargs)
    except TypeError as e:
        if "compute_weight_metrics" in str(e):
            kwargs.pop("compute_weight_metrics", None)
            strat, res, row = fn(**kwargs)
        else:
            raise
    return row


yearly_rows = []
for yr in WF_YEARS:
    bt_i, clf_i, train_df_i, cutoff_i, start_i, end_i = _build_bt_panel_for_year_anchored(test_year=yr, use_cache=True)

    tmp = []
    for case in ABLT_CASES_YEARLY:
        row = _run_case_on_panel(clf_model=clf_i, panel_obj=bt_i, case=case)
        row.update({
            "test_year": yr,
            "train_cutoff": cutoff_i.date().isoformat(),
            "train_rows": int(len(train_df_i)),
        })
        tmp.append(row)

    df_year = pd.DataFrame(tmp).sort_values(["sharpe", "total_return_pct"], ascending=[False, False]).reset_index(drop=True)
    df_year["rank_in_year"] = np.arange(1, len(df_year) + 1)
    yearly_rows.append(df_year)

ablation_yearly_df = pd.concat(yearly_rows, axis=0, ignore_index=True)

print(f"Per-year top configs (FAST_ABLATION_METRICS={FAST_ABLATION_METRICS}):")
for yr in WF_YEARS:
    print(f"\nYear {yr} top 10:")
    display(ablation_yearly_df.loc[ablation_yearly_df["test_year"] == yr, [
        "rank_in_year", "case", "total_return_pct", "sharpe", "max_drawdown_pct", "avg_turnover"
    ]].head(10))

stability = (
    ablation_yearly_df
    .groupby("case", as_index=False)
    .agg(
        mean_rank=("rank_in_year", "mean"),
        std_rank=("rank_in_year", "std"),
        mean_sharpe=("sharpe", "mean"),
        std_sharpe=("sharpe", "std"),
        mean_return_pct=("total_return_pct", "mean"),
        std_return_pct=("total_return_pct", "std"),
        mean_mdd_pct=("max_drawdown_pct", "mean"),
    )
    .sort_values(["mean_rank", "mean_sharpe"], ascending=[True, False])
    .reset_index(drop=True)
)

print("Stability summary across years (top 30):")
display(stability.head(30))

wide_sharpe = ablation_yearly_df.pivot(index="case", columns="test_year", values="sharpe")
print("Sharpe by case/year (top 30 by mean sharpe):")
top_cases = stability.sort_values("mean_sharpe", ascending=False).head(30)["case"].tolist()
display(wide_sharpe.loc[[c for c in top_cases if c in wide_sharpe.index]])
